# 1 · Introduction au machine learning et au deep learning — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch01_introduction/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des ✅/❌ pour les expériences). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch01_introduction/03_notebook.ipynb`, créée par `python tools/start_chapter.py 1`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 1.1 à 1.4 | vérifier tes exercices papier ✏️ | ✏️ | ★ | |
| A | 1.9 | Penguins : échantillons, features et labels | 📦 | ★ | 10 |
|  | 1.10 | MNIST : une image, 784 nombres | 📦 | ★ | 10 |
|  | 1.11 | Holmes et Verne : le texte devient des nombres | 📦 | ★★ | 15 |
|  | 1.12 | Taches solaires : tracer, lisser, repérer le cycle | 📦 | ★★ | 20 |
|  | 1.13 | Lire les data cards des quatre fils rouges | 🛠️ | ★★ | 15 |
| B | 1.14 | Mémoriser n'est pas apprendre | 🔮 | ★★ | 15 |
|  | 1.15 | Un système expert pour les manchots | 🔨 | ★★ | 20 |
|  | 1.16 | La boucle d'entraînement à la main | 🔨 | ★★ | 30 |
|  | 1.17 | Learning rate : trop prudent, trop pressé | 🔬 | ★★ | 20 |
|  | 1.18 | Un arbre de décision apprend les règles à ta place | 📦 | ★★ | 20 |
|  | 1.19 | Un manchot d'une espèce jamais vue | 🔮 | ★★ | 15 |
|  | 1.20 | Le score trop beau pour être vrai | 🐛 | ★★ | 20 |
| C | 1.21 | Regrouper les manchots sans leurs étiquettes | 📦 | ★★ | 20 |
|  | 1.22 | L'agent cuisinier : apprendre par la récompense | 🔬 | ★★ | 30 |
|  | 1.23 | Un réseau de neurones en boîte noire sur MNIST | 📦 | ★★ | 25 |
|  | 1.24 | Fabriquer du faux Holmes et du faux Verne | 🔨 | ★★★ | 35 |
| D | 1.25 | Battre l'expert : 95 % avec tes propres règles | 🏆 | ★★★ | 40 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="1")  # your own library

## Objectifs et rappel express

- Décrire les quatre fils rouges du workbook avec le vocabulaire du ML : échantillons, features, labels.
- Voir la différence entre mémoriser et généraliser, et évaluer honnêtement sur un jeu de test.
- Écrire des règles à la main, puis une vraie boucle d'entraînement, et comparer avec un modèle qui apprend seul ses règles.
- Faire tourner en boîte noire un clustering, un agent qui apprend par la récompense, un réseau de neurones et un générateur de texte.

**Rappel express.** Un dataset est un tableau : une ligne par **échantillon**, une colonne par **feature**, plus le **label** à prédire. Entraîner : prédire, mesurer l'erreur (la **loss**), corriger les **paramètres** d'un pas réglé par le **learning rate**. On mesure la **généralisation** sur un **jeu de test** mis de côté, jamais utilisé pour apprendre. $\text{accuracy} = \frac{\text{prédictions correctes}}{\text{nombre d'échantillons}}$. En scikit-learn : `model.fit(X_train, y_train)`, puis `model.predict(X_new)` ou `model.score(X_test, y_test)`.

## Partie 0 · Vérifier tes exercices papier (✏️ de 1.1 à 1.4)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125` ou `[7, -2]`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande. Les réponses pas encore remplies affichent ⏳.

**Ex 1.1 — Accuracy et erreurs à l'échelle d'un centre de tri**

In [ ]:
# Ex 1.1: the values you found on paper
answer_1_1a = ...  # a) the accuracy of the book's network, a proportion (4 decimals)
answer_1_1b = ...  # b) its error rate, in % (2 decimals)
answer_1_1c = ...  # c) the number of digits misread per day
answer_1_1d = ...  # d) P(a 5-digit code is read without any error), 4 decimals
answer_1_1e = ...  # e) the number of wrong codes among 240 000 envelopes

for letter, answer in zip("abcde", [answer_1_1a, answer_1_1b, answer_1_1c, answer_1_1d, answer_1_1e]):
    wb.check(f"1.1{letter}", answer)

**Ex 1.2 — Concerts : la valeur manquante et celle de demain**

In [ ]:
# Ex 1.2: the values you found on paper
answer_1_2a = ...  # a) the attendance of May 8, by linear interpolation between May 7 and May 9
answer_1_2b = ...  # b) the slope of the line through the first and the last day (people per day)
answer_1_2c = ...  # c) the value of this line on May 8
answer_1_2d = ...  # d) the prediction of this line for May 13
answer_1_2e = ...  # e) the prediction for May 13 from the last two days only
answer_1_2f = ...  # f) the band's income on May 13, in euros (with the prediction of d)

for letter, answer in zip("abcdef", [answer_1_2a, answer_1_2b, answer_1_2c, answer_1_2d, answer_1_2e, answer_1_2f]):
    wb.check(f"1.2{letter}", answer)

**Ex 1.3 — Compter les connexions d'un réseau en couches**

In [ ]:
# Ex 1.3: the values you found on paper
answer_1_3a = ...  # a) the number of connections of the 4 -> 3 -> 2 network
answer_1_3b = ...  # b) its number of parameters (weights + biases)
answer_1_3c = ...  # c) the number of parameters of the 784 -> 128 -> 10 network
answer_1_3d = ...  # d) the same with 256 hidden neurons
answer_1_3e = ...  # e) the number of parameters of the 784 -> 128 -> 128 -> 10 network

for letter, answer in zip("abcde", [answer_1_3a, answer_1_3b, answer_1_3c, answer_1_3d, answer_1_3e]):
    wb.check(f"1.3{letter}", answer)

**Ex 1.4 — Moins de nombres pour dire la même chose**

In [ ]:
# Ex 1.4: the values you found on paper
answer_1_4a = ...  # a) the number of features that carry information
answer_1_4b = ...  # b) 68 kg in pounds (1 decimal)
answer_1_4c = ...  # c) the number of independent pieces of information in (kg, lb)
answer_1_4d = ...  # d) the distance of P from the start of the road, in m
answer_1_4e = ...  # e) the map coordinates of the car at 750 m, a list
answer_1_4f = ...  # f) the position of Q along the road, in m
answer_1_4g = ...  # g) the distance lost (from Q to the centre line), in m

for letter, answer in zip("abcdefg", [answer_1_4a, answer_1_4b, answer_1_4c, answer_1_4d, answer_1_4e, answer_1_4f, answer_1_4g]):
    wb.check(f"1.4{letter}", answer)

## Partie A · Les quatre fils rouges du workbook

Fiche §1.2.2 et « Les quatre fils rouges ». Tu ouvres les quatre datasets qui reviendront tout au long du workbook et tu les regardes avec le vocabulaire du chapitre : échantillons, features, labels, et le type de problème que chacun illustre.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")

### Ex 1.9 — Penguins : échantillons, features et labels 📦 ★ ⏱️ 10 min
**Objectif :** décrire un dataset tabulaire avec le vocabulaire du machine learning.  
**Prérequis :** 0A (pandas) · fiche §1.2.2 · livre §1.2.2, §1.3.1 · **Fil rouge :** Penguins · **Parcours :** R, C

Le dataset Palmer Penguins (déjà vu en 0A) décrit des manchots de trois espèces. Avec le vocabulaire du livre (§1.2.2) : chaque **ligne** est un **échantillon**, chaque **colonne** une **feature**, et la colonne que l'on cherche à prédire est le **label**. Ici, le label est l'espèce (`species`).

a) `n_samples` : le nombre d'échantillons de `penguins_all`.
b) `n_features` : le nombre de features qui décrivent chaque manchot quand le label est `species` (toutes les **autres** colonnes).
c) `species_counts` : le nombre de manchots de chaque espèce, **du plus fréquent au moins fréquent**, sous forme d'une liste de 3 entiers (`value_counts()`, puis `.tolist()`).
d) `n_complete` : le nombre de manchots qui n'ont **aucune** valeur manquante (`dropna()`).
e) `first_vector` : les 4 mesures du premier manchot, dans l'ordre de `MEASURES`, sous forme d'une liste (`penguins_all.loc[0, MEASURES]`). Ce manchot est désormais un **vecteur** (0B).
f) `label_for_mass` : si l'on voulait prédire la **masse** d'un manchot à partir de ses autres colonnes, quel serait le nom de la colonne label ?

In [ ]:
penguins_all = wb.datasets.load_penguins()     # all the penguins, missing values included
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins_all.head()

In [ ]:
n_samples = ...        # a)
n_features = ...       # b)
species_counts = ...   # c) a list of 3 integers, most frequent species first
n_complete = ...       # d)
first_vector = ...     # e) a list of 4 numbers
label_for_mass = ...   # f) a column name (a string)

In [ ]:
wb.check("1.9a", n_samples)
wb.check("1.9b", n_features)
wb.check("1.9c", species_counts)
wb.check("1.9d", n_complete)
wb.check("1.9e", first_vector)
wb.check("1.9f", label_for_mass)

### Ex 1.10 — MNIST : une image, 784 nombres 📦 ★ ⏱️ 10 min
**Objectif :** voir qu'une image n'est, pour un ordinateur, qu'un tableau de nombres.  
**Prérequis :** 0A (NumPy, `wb.plot.show_images`) · fiche « Les quatre fils rouges » · livre §1.1.1, §1.7 · **Fil rouge :** MNIST · **Parcours :** R, C

MNIST contient 70 000 chiffres écrits à la main (60 000 pour l'entraînement, 10 000 pour le test), en niveaux de gris : chaque pixel est un entier, d'autant plus grand que l'encre est foncée. Le livre y revient souvent (§1.7), le workbook aussi.

a) `shape_mnist` : la forme de `X_mnist` (un tuple).
b) `n_pixels` : le nombre de pixels d'**une** image.
c) `pixel_range` : la liste `[plus petite valeur, plus grande valeur]` des pixels de tout `X_mnist`.
d) `first_label` : le label de la première image.
e) `n_ink` : le nombre de pixels **non nuls** de la première image (un masque booléen, 0A).
f) `most_common_digit` : le chiffre le plus fréquent parmi les labels `y_mnist` (`np.bincount`, puis `argmax`).

La vérification affiche ensuite 16 autres images avec leur label, et un morceau de la première image **en nombres**.

In [ ]:
X_mnist, y_mnist = wb.datasets.load_mnist("train")   # 60 000 training images and their labels
print(type(X_mnist), X_mnist.dtype)

In [ ]:
shape_mnist = ...         # a)
n_pixels = ...            # b)
pixel_range = ...         # c) [min, max]
first_label = ...         # d)
n_ink = ...               # e)
most_common_digit = ...   # f)

In [ ]:
wb.check("1.10a", shape_mnist)
wb.check("1.10b", n_pixels)
wb.check("1.10c", pixel_range)
wb.check("1.10d", first_label)
wb.check("1.10e", n_ink)
wb.check("1.10f", most_common_digit)
wb.plot.show_images(X_mnist[1:17], y_mnist[1:17], ncols=8)   # images 1 to 16 (the first one is yours to read)
plt.show()
print(X_mnist[0][4:24, 4:24])   # the centre of the first image, as numbers

### Ex 1.11 — Holmes et Verne : le texte devient des nombres 📦 ★★ ⏱️ 15 min
**Objectif :** transformer un texte en nombres (codes des caractères, fréquences des lettres) et comparer deux textes avec une distance.  
**Prérequis :** 0A (chaînes, `Counter`) · 0B (distance entre vecteurs) · fiche « Les quatre fils rouges » · livre §1.1.1 · **Fil rouge :** Holmes/Verne · **Parcours :** R, M, C

Deux romans du domaine public servent de fils rouges « texte » : *The Adventures of Sherlock Holmes* (anglais) et *Le Tour du monde en quatre-vingts jours* (français). Pour un ordinateur, un texte est une suite de caractères, et chaque caractère a un numéro (son code Unicode, `ord`).

a) `n_chars` : la liste `[nombre de caractères de holmes, nombre de caractères de verne]`.
b) `codes` : la liste des codes (`ord`) des 4 caractères de la chaîne `"Aa é"` (l'espace compte).
c) `n_distinct` : la liste `[nombre de caractères différents dans holmes, … dans verne]` (`set`). Pourquoi le français en a-t-il davantage ?
d) et e) `letter_freq(text)` : le vecteur des **26 fréquences** des lettres de `a` à `z`. On passe le texte en minuscules (`text.lower()`), on ne garde que les caractères entre `"a"` et `"z"` (les lettres accentuées ne comptent pas), puis la fréquence d'une lettre est son nombre d'apparitions divisé par le nombre total de lettres gardées. La vérification regarde d) `letter_freq(holmes)` et e) `letter_freq(verne)` (3 décimales).
f) `distances` : la liste `[d_halves, d_languages]` (4 décimales), où `d_halves` est la distance (norme de la différence, 0B) entre les vecteurs de fréquences des deux moitiés de `holmes` (`holmes[:len(holmes) // 2]` et `holmes[len(holmes) // 2:]`), et `d_languages` la distance entre les vecteurs de `holmes` et de `verne`. Qu'en conclus-tu ?

In [ ]:
holmes = wb.datasets.load_holmes()     # a str: the whole book, in English
verne = wb.datasets.load_verne()       # a str: the whole book, in French
print(holmes[:300])
print("-" * 60)
print(verne[:300])

In [ ]:
n_chars = ...      # a)
codes = ...        # b)
n_distinct = ...   # c)


def letter_freq(text):
    """The 26 frequencies of the letters a to z among the letters a-z of text (in lower case), a NumPy array."""
    raise NotImplementedError("letter_freq() is not written yet")


distances = ...    # f) [d_halves, d_languages]

In [ ]:
wb.check("1.11a", n_chars)
wb.check("1.11b", codes)
wb.check("1.11c", n_distinct)
with wb.attempt("1.11d"):
    wb.check("1.11d", letter_freq(holmes))
    wb.check("1.11e", letter_freq(verne))
    letters = [chr(k) for k in range(ord("a"), ord("z") + 1)]
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar(np.arange(26) - 0.2, letter_freq(holmes), width=0.4, label="Holmes (EN)")
    ax.bar(np.arange(26) + 0.2, letter_freq(verne), width=0.4, label="Verne (FR)")
    ax.set_xticks(np.arange(26), letters)
    ax.set_ylabel("frequency")
    ax.legend()
    plt.show()
wb.check("1.11f", distances)

### Ex 1.12 — Taches solaires : tracer, lisser, repérer le cycle 📦 ★★ ⏱️ 20 min
**Objectif :** explorer une série temporelle réelle, la débruiter par une moyenne mobile et mesurer son cycle.  
**Prérequis :** 0A (pandas) · 0B.4 (moyenne mobile) · fiche §1.3.2, §1.4.2 · livre §1.3.2, §1.4.2 · **Fil rouge :** taches solaires · **Parcours :** C

Le nombre mensuel de taches solaires a été reconstitué depuis 1749 et il est tenu à jour par le SILSO, à l'observatoire royal de Belgique. La série est très bruitée, mais cache un cycle célèbre. Deux idées du livre s'y rencontrent : le **débruitage** (§1.4.2) et la **régression** (§1.3.2 : prévoir la valeur du mois suivant, que tu feras au ch. 22).

a) `n_months` : le nombre de mois de la série `sun` (déjà limitée aux mois définitifs).
b) `record_year` : l'année du mois qui compte le plus de taches (`idxmax`, puis la colonne `year`).
c) `smooth` : la moyenne mobile sur 13 mois, **exactement** `sun["sunspots"].rolling(13).mean()` (la moyenne du mois et des 12 précédents, 0B.4) ; puis `n_missing`, le nombre de valeurs manquantes (`NaN`) de `smooth`. D'où viennent-elles ?
d) `peak_year` : l'année où la série **lissée** est la plus haute.
e) `peaks` : les positions des sommets de la série lissée, données par l'outil fourni `local_maxima(smooth.to_numpy(), 60)` (un mois est un sommet s'il est le plus haut à 60 mois près, avant comme après) ; puis `n_peaks`, leur nombre.
f) `mean_period` : la durée moyenne, en années (1 décimale), entre deux sommets successifs ; utilise la colonne `decimal_year` aux positions `peaks` et `np.diff`.

La vérification trace la série brute, la série lissée et les sommets.

In [ ]:
sun = wb.datasets.load_sunspots(definitive_only=True)   # monthly sunspot numbers, final values only
print(sun[["date", "sunspots"]].head(3))


def local_maxima(values, half_window):
    """Positions t where values[t] is the largest value of values[t - half_window : t + half_window + 1]."""
    positions = []
    for t in range(half_window, len(values) - half_window):
        window = values[t - half_window: t + half_window + 1]
        if not np.isnan(window).any() and values[t] == window.max():
            positions.append(t)
    return positions

In [ ]:
n_months = ...      # a)
record_year = ...   # b)
smooth = ...        # c) the 13-month moving average
n_missing = ...     # c)
peak_year = ...     # d)
peaks = ...         # e) a list of positions
n_peaks = ...       # e)
mean_period = ...   # f) in years

In [ ]:
wb.check("1.12a", n_months)
wb.check("1.12b", record_year)
wb.check("1.12c", n_missing)
wb.check("1.12d", peak_year)
wb.check("1.12e", n_peaks)
wb.check("1.12f", mean_period)
if smooth is not ... and peaks is not ...:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(sun["decimal_year"], sun["sunspots"], color="0.75", lw=0.6, label="monthly")
    ax.plot(sun["decimal_year"], smooth, lw=1.8, label="13-month moving average")
    ax.plot(sun["decimal_year"].to_numpy()[peaks], smooth.to_numpy()[peaks], "o", label="peaks")
    ax.set_xlabel("year")
    ax.set_ylabel("sunspots")
    ax.legend(loc="upper left")
    plt.show()

### Ex 1.13 — Lire les data cards des quatre fils rouges 🛠️ ★★ ⏱️ 15 min
**Objectif :** lire la fiche d'un dataset (provenance, licence, biais, limites) avant de s'en servir.  
**Prérequis :** Ex 1.9 à 1.12 · `data/cards/` · livre §1.8 · **Parcours :** C

Un professionnel ne se sert jamais d'un dataset sans lire sa **data card** (*fiche de données*) : d'où viennent les données, sous quelle licence, quels biais et quelles limites. Le workbook en a une par dataset, dans `data/cards/`. La cellule suivante affiche le catalogue et les fiches des quatre fils rouges.

Réponds dans la cellule **📝 Mes réponses** :
1. Pour chacun des cinq datasets (Penguins, MNIST, Holmes, Verne, taches solaires) : sa **licence**, et peux-tu t'en servir dans un **produit vendu** ?
2. Quelle limite de Penguins pourrait pousser un modèle à « tricher » en regardant une autre colonne que la morphologie ?
3. Pourquoi la fiche de MNIST dit-elle qu'un modèle médiocre peut y paraître bon ?
4. En quoi Holmes et Verne diffèrent-ils des textes sur lesquels on entraîne un LLM ?
5. Quelles précautions prendre pour les taches solaires avant d'enregistrer des résultats (colonne `definitive`) ?

In [ ]:
print(wb.datasets.list_datasets()[["dataset", "type", "versionné", "fiche"]].to_string(index=False))
for name in ["penguins", "mnist", "holmes", "verne", "sunspots"]:
    print("=" * 80)
    wb.datasets.dataset_card(name)

📝 **Mes réponses** (1.13) :

1. …
2. …
3. …
4. …
5. …

## Partie B · Apprendre à partir d'exemples étiquetés

Fiche §1.1.2 et §1.2. Les 333 manchots complets sont tirés au sort une fois pour toutes : 233 pour apprendre (le **jeu d'entraînement**) et 100 mis de côté (le **jeu de test**), qui ne servent qu'à mesurer si ce qui a été appris se **généralise**. Tu compares un modèle qui apprend par cœur, des règles écrites à la main, une vraie boucle d'entraînement et un modèle qui apprend ses règles tout seul.

In [ ]:
# Penguins, split once and for all: a training set and a test set set aside
penguins = wb.datasets.load_penguins(dropna=True)     # the 333 complete penguins
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
rng_split = np.random.default_rng(42)                  # the random draw of the penguins set aside
order = rng_split.permutation(len(penguins))
test = penguins.iloc[order[:100]].reset_index(drop=True)    # 100 penguins set aside: the test set
train = penguins.iloc[order[100:]].reset_index(drop=True)   # 233 penguins to learn from: the training set
X_train, y_train = train[MEASURES], train["species"]
X_test, y_test = test[MEASURES], test["species"]


def accuracy(y_true, y_pred):
    """Proportion of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def predict_with(rule, X):
    """Apply rule(bill_length, bill_depth, flipper_length, body_mass) to every row of X."""
    return np.array([rule(*row) for row in X[MEASURES].to_numpy()])


print(f"training set: {len(train)} penguins · test set: {len(test)} penguins")

### Ex 1.14 — Mémoriser n'est pas apprendre 🔮 ★★ ⏱️ 15 min
**Objectif :** voir qu'un score parfait sur les exemples appris ne dit rien de la généralisation.  
**Prérequis :** Ex 1.9 · fiche §1.2.1, §1.2.3 · livre §1.2.1 à §1.2.3 · **Fil rouge :** Penguins · **Parcours :** R, C

Le `Memorizer` ci-dessous apprend « par cœur », comme l'école absurde du livre (§1.2.1) : pendant l'entraînement (`fit`), il range chaque manchot dans une table (ses 4 mesures → son espèce). Pour prédire (`predict`), il cherche les mesures du manchot dans sa table ; s'il ne les trouve pas, il répond l'espèce la plus fréquente de l'entraînement. Il a la même interface que les modèles de scikit-learn : `fit`, puis `predict` (encadré 🧮 de la fiche).

**Sans rien exécuter**, classe l'accuracy qu'il obtiendra dans une catégorie : `"parfaite"` (100 %), `"bonne"` (de 80 % à 100 % exclu), `"moyenne"` (de 50 % à 80 % exclu) ou `"mauvaise"` (moins de 50 %).

a) sur le **jeu d'entraînement** (les manchots qu'il a appris) ;
b) sur le **jeu de test** (100 manchots qu'il n'a jamais vus).

Écris ton hypothèse (cellule 📝), puis tes deux prédictions ; la vérification ne regarde que tes prédictions. Ensuite seulement, exécute l'**Expérience**.

In [ ]:
class Memorizer:
    """Learns by heart: a table (4 measures -> species). Unknown penguin: the most frequent species."""

    def fit(self, X, y):
        self.table_ = {tuple(row): label for row, label in zip(X.to_numpy(), y)}
        self.default_ = y.value_counts().idxmax()
        return self

    def predict(self, X):
        return np.array([self.table_.get(tuple(row), self.default_) for row in X.to_numpy()])

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_1_14a = ...   # a) on the training set: "parfaite", "bonne", "moyenne" or "mauvaise"
prediction_1_14b = ...   # b) on the test set

In [ ]:
wb.check("1.14a", prediction_1_14a)
wb.check("1.14b", prediction_1_14b)

**Expérience** : exécute la cellule et compare avec tes prédictions.

In [ ]:
memorizer = Memorizer().fit(X_train, y_train)
print("accuracy on the training set:", accuracy(y_train, memorizer.predict(X_train)))
print("accuracy on the test set:    ", accuracy(y_test, memorizer.predict(X_test)))

c) `n_seen` : combien de manchots du jeu de test figurent dans la table du mémoriseur (`tuple(row) in memorizer.table_`) ?  
d) `default_species` : l'espèce qu'il répond alors pour eux. Compare l'accuracy sur le test avec la part de cette espèce dans `y_test`.

In [ ]:
n_seen = ...            # c)
default_species = ...   # d)

In [ ]:
wb.check("1.14c", n_seen)
wb.check("1.14d", default_species)

### Ex 1.15 — Un système expert pour les manchots 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer des règles d'expert, mesurer leur accuracy et repérer les cas qu'elles ratent.  
**Prérequis :** Ex 1.9 · 0A (`if`/`elif`, `pd.crosstab`) · fiche §1.1.2 · livre §1.1.2 · **Fil rouge :** Penguins · **Parcours :** C

Une biologiste te donne ses règles pour reconnaître l'espèce d'un manchot :
1. « Un manchot de plus de 4 700 g est un Gentoo. »
2. « Sinon, un bec de plus de 45 mm de long signe un Chinstrap. »
3. « Sinon, c'est un Adélie. »

C'est un petit **système expert** (§1.1.2) : des règles écrites à la main, sans rien apprendre des données.

Écris `expert_rule(bill_length, bill_depth, flipper_length, body_mass)`, qui renvoie `"Gentoo"`, `"Chinstrap"` ou `"Adelie"` en appliquant **exactement** ces trois règles (« plus de » : strictement plus grand). L'outil fourni `predict_with(expert_rule, X)` l'applique à chaque manchot d'un tableau.

a) `expert_train_acc` : l'accuracy de ces règles sur le jeu d'entraînement (3 décimales).
b) `expert_test_acc` : leur accuracy sur le jeu de test (2 décimales).
c) `worst_species` : l'espèce dont les manchots d'entraînement sont **le plus souvent** mal classés (regarde `pd.crosstab(y_train, predictions)`, affiché par la vérification). Pourquoi ces manchots-là sont-ils mal classés ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for species in ["Adelie", "Chinstrap", "Gentoo"]:
    part = train[train["species"] == species]
    axes[0].scatter(part["body_mass_g"], part["bill_length_mm"], s=14, label=species)
    axes[1].scatter(part["flipper_length_mm"], part["bill_length_mm"], s=14, label=species)
axes[0].axvline(4700, color="0.3", ls="--")
axes[0].axhline(45, color="0.3", ls=":")
axes[0].set_xlabel("body_mass_g")
axes[1].set_xlabel("flipper_length_mm")
for ax in axes:
    ax.set_ylabel("bill_length_mm")
    ax.legend()
fig.suptitle("Training set: the expert's thresholds (dashed: 4 700 g, dotted: 45 mm)")
plt.show()

In [ ]:
def expert_rule(bill_length, bill_depth, flipper_length, body_mass):
    """The biologist's three rules: returns "Gentoo", "Chinstrap" or "Adelie"."""
    raise NotImplementedError("expert_rule() is not written yet")


expert_train_acc = ...   # a)
expert_test_acc = ...    # b)
worst_species = ...      # c)

In [ ]:
with wb.attempt("1.15"):
    print(pd.crosstab(y_train, predict_with(expert_rule, X_train), rownames=["true"], colnames=["predicted"]))
wb.check("1.15a", expert_train_acc)
wb.check("1.15b", expert_test_acc)
wb.check("1.15c", worst_species)

### Ex 1.16 — La boucle d'entraînement à la main 🔨 ★★ ⏱️ 30 min
**Objectif :** coder une boucle d'entraînement complète (prédire, mesurer l'erreur, corriger) pour une droite.  
**Prérequis :** 0B (fonction affine, loss, 101.5.3) · Rappel 1.R3 · fiche §1.2.2 · livre §1.2.2, §1.2.4 (fig. 1.8) · **Fil rouge :** synthétique · **Parcours :** R, M, C

Les points `(x_line, y_line)` suivent à peu près une droite. Le modèle est une droite $\hat{y} = w\,x + b$ : ses deux **paramètres** sont $w$ et $b$. Pour l'entraîner, on part de $w = b = 0$, puis on montre les 50 échantillons **un par un, dans l'ordre** ; pour chacun, on fait l'étape de la figure 1.8 du livre :
1. **prédire** $\hat{y} = w\,x + b$ ;
2. **comparer** : l'erreur est $y - \hat{y}$ ;
3. **corriger** les paramètres, avec la règle (admise ici, justifiée aux ch. 5 et 19) :
$$w \leftarrow w + \eta\,(y - \hat{y})\,x \qquad b \leftarrow b + \eta\,(y - \hat{y})$$
où $\eta$ est le **learning rate**. Un passage complet sur les 50 échantillons est une **epoch**. Pour suivre l'entraînement, on calcule après chaque epoch la **loss** : l'erreur quadratique moyenne $L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$ sur tous les échantillons (0B, 101.1.3).

Écris les quatre fonctions, puis vérifie :
a) `mse(0, 0, x_line, y_line)` : la loss avant tout entraînement (3 décimales) ;
b) `train_step(0.0, 0.0, x_line[0], y_line[0], 0.01)` : les nouveaux $(w, b)$ après **une seule** correction (4 décimales) ;
c) $(w, b)$ après **une** epoch avec $\eta = 0{,}01$ (4 décimales) ;
d) $(w, b)$ après **20** epochs avec $\eta = 0{,}01$ (3 décimales) ;
e) la loss après ces 20 epochs (4 décimales).

La vérification trace la droite après 0, 1 et 20 epochs, et la loss epoch par epoch. Les données ont été fabriquées avec $w = 2$ et $b = 1$, plus du bruit : retrouves-tu ces valeurs ?

In [ ]:
X_line, y_line = wb.synth.make_linear(n=50, w=(2.0,), b=1.0, noise=0.5, seed=0)   # y ≈ 2x + 1 + noise
x_line = X_line[:, 0]                                                              # a 1-D array of 50 numbers
print(np.round(x_line[:3], 3), np.round(y_line[:3], 3))

In [ ]:
def predict_line(w, b, x):
    """The prediction y_hat = w x + b (x can be a number or a NumPy array)."""
    raise NotImplementedError("predict_line() is not written yet")


def mse(w, b, x, y):
    """Mean squared error of the line (w, b) on the samples (x, y)."""
    raise NotImplementedError("mse() is not written yet")


def train_step(w, b, x_i, y_i, eta):
    """One correction on one sample (x_i, y_i): returns the new (w, b)."""
    raise NotImplementedError("train_step() is not written yet")


def train_line(x, y, eta, n_epochs):
    """Start from w = b = 0, then n_epochs passes over the samples, in order.

    Returns (w, b, losses): losses[k] is the MSE after k epochs (losses[0]: before training).
    """
    raise NotImplementedError("train_line() is not written yet")

In [ ]:
with wb.attempt("1.16"):
    wb.check("1.16a", mse(0, 0, x_line, y_line))
    wb.check("1.16b", list(train_step(0.0, 0.0, x_line[0], y_line[0], 0.01)))
    wb.check("1.16c", list(train_line(x_line, y_line, 0.01, 1)[:2]))
    w_16, b_16, losses_16 = train_line(x_line, y_line, 0.01, 20)
    wb.check("1.16d", [w_16, b_16])
    wb.check("1.16e", losses_16[-1])
    verdict("1.16", len(losses_16) == 21 and abs(losses_16[0] - mse(0, 0, x_line, y_line)) < 1e-9,
            "losses contient bien la loss avant l'entraînement, puis une loss par epoch (21 valeurs).",
            "losses doit contenir 21 valeurs : la loss AVANT l'entraînement (w = b = 0), puis une par epoch.")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].scatter(x_line, y_line, s=14, color="0.5")
    grid = np.linspace(-3, 3, 2)
    for epochs in (0, 1, 20):
        w, b, _ = train_line(x_line, y_line, 0.01, epochs)
        axes[0].plot(grid, predict_line(w, b, grid), label=f"after {epochs} epoch(s)")
    axes[0].legend()
    axes[1].plot(losses_16, marker="o")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("loss (MSE)")
    plt.show()

### Ex 1.17 — Learning rate : trop prudent, trop pressé 🔬 ★★ ⏱️ 20 min
**Objectif :** observer l'effet du learning rate sur la vitesse et la stabilité de l'entraînement.  
**Prérequis :** Ex 1.16 · fiche §1.2.2 · livre §1.2.2, §1.2.4 · **Fil rouge :** synthétique · **Parcours :** C

Reprends `train_line` de 1.16 et entraîne la droite pendant 20 epochs avec quatre learning rates : $\eta = 0{,}001$, $0{,}01$, $0{,}1$ et $0{,}5$.

Écris `losses_by_eta()`, qui renvoie le dictionnaire `{eta: liste des losses}` pour les quatre valeurs de `ETAS_17` (une ligne de compréhension suffit). La vérification trace la loss epoch par epoch : à gauche les quatre courbes (échelle logarithmique), à droite les trois premières seules, pour les voir de près ; elle affiche aussi, pour chaque $\eta$, la loss après 1 et après 20 epochs, et la valeur finale de $w$. Réponds ensuite dans la cellule **📝 Mes réponses** :
1. Quel learning rate est « trop prudent » ? À quoi le vois-tu ?
2. Que se passe-t-il avec $\eta = 0{,}5$ ? Que vaut $w$ après 20 epochs ?
3. Entre $0{,}01$ et $0{,}1$, lequel fait baisser la loss le plus vite au début ? Lequel finit le plus bas ?
4. Le livre propose de réduire le learning rate au fil de l'entraînement (§1.2.2). Pourquoi est-ce une bonne idée, d'après tes courbes ?

In [ ]:
ETAS_17 = [0.001, 0.01, 0.1, 0.5]


def plot_losses_17(results):
    """Left: the four loss curves (log scale). Right: the first three only (a closer look)."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, etas in zip(axes, [ETAS_17, ETAS_17[:3]]):
        for eta in etas:
            ax.plot(results[eta], marker="o", ms=3, label=f"eta = {eta}")
        ax.set_yscale("log")
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss (MSE, log scale)")
        ax.legend()
    axes[1].set_title("without eta = 0.5")
    plt.show()

In [ ]:
def losses_by_eta():
    """A dict {eta: the list of losses of train_line(x_line, y_line, eta, 20)} for every eta of ETAS_17."""
    raise NotImplementedError("losses_by_eta() is not written yet")

In [ ]:
with wb.attempt("1.17"):
    results_17 = losses_by_eta()
    plot_losses_17(results_17)
    for eta in ETAS_17:
        w, b, losses = train_line(x_line, y_line, eta, 20)
        print(f"eta = {eta:<6} loss after 1 epoch: {losses[1]:10.4g}   after 20: {losses[-1]:10.4g}   final w = {w:.4g}")
    verdict("1.17", results_17[0.001][-1] > results_17[0.01][-1] and results_17[0.5][-1] > 1e6,
            "tes courbes montrent bien un learning rate trop prudent et un autre qui fait exploser la loss.",
            "tes courbes ne ressemblent pas à celles attendues : vérifie train_line (1.16) et le dictionnaire.")

📝 **Mes réponses** (1.17) :

1. …
2. …
3. …
4. …

### Ex 1.18 — Un arbre de décision apprend les règles à ta place 📦 ★★ ⏱️ 20 min
**Objectif :** entraîner un modèle de scikit-learn en boîte noire, l'évaluer sur le jeu de test et lire les règles qu'il a apprises.  
**Prérequis :** Ex 1.9, Ex 1.14 · fiche 🧮 « scikit-learn en boîte noire » · livre §1.1.2, §1.2.3, §1.3.1 · **Fil rouge :** Penguins · **Parcours :** R, C

Au lieu d'écrire des règles, laissons un algorithme les **apprendre** à partir des exemples d'entraînement. Un **arbre de décision** (détaillé au ch. 13) pose des questions du type « bec ≤ 40 mm ? » et les choisit lui-même. Les trois lignes de scikit-learn sont fournies : on crée le modèle en fixant ses **hyperparamètres** (ici, au plus deux questions à la suite : `max_depth=2`), puis `fit` apprend ses **paramètres** (les questions et les seuils) sur le jeu d'entraînement.

a) `tree_train_acc` : son accuracy sur le jeu d'entraînement (`tree.score(X, y)`, 3 décimales).
b) `tree_test_acc` : son accuracy sur le jeu de test (2 décimales).
c) `root_feature` : le nom de la feature de la **première** question de l'arbre (lis `rules`, affiché par la cellule fournie).
d) `root_threshold` : le seuil de cette première question (1 décimale).
e) `gain_vs_expert` : combien de manchots du jeu de test l'arbre classe-t-il correctement **en plus** des règles de la biologiste (1.15) ? (un entier)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # hyperparameters: chosen by you
tree.fit(X_train, y_train)                                    # parameters: learned from the training set
rules = export_text(tree, feature_names=MEASURES)
print(rules)

In [ ]:
tree_train_acc = ...   # a)
tree_test_acc = ...    # b)
root_feature = ...     # c) a column name
root_threshold = ...   # d)
gain_vs_expert = ...   # e) an integer

In [ ]:
wb.check("1.18a", tree_train_acc)
wb.check("1.18b", tree_test_acc)
wb.check("1.18c", root_feature)
wb.check("1.18d", root_threshold)
wb.check("1.18e", gain_vs_expert)

### Ex 1.19 — Un manchot d'une espèce jamais vue 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir, en lisant ses règles, ce que répond un classifieur face à un cas hors de ses classes.  
**Prérequis :** Ex 1.18 · fiche §1.3.1 · livre §1.3.1 (fig. 1.10) · **Fil rouge :** Penguins · **Parcours :** C

L'arbre de 1.18 ne connaît que trois espèces. Deux visiteurs se présentent :
- un **manchot empereur**, la plus grande espèce (jusqu'à environ 40 kg, un bec d'environ 8 cm) : on prend un bec de 80 mm de long, une masse de 30 000 g, et, pour les deux mesures que personne n'a relevées ici, des valeurs **imaginées** pour l'exercice : 22 mm d'épaisseur de bec et 340 mm de nageoire ;
- un **poussin imaginaire**, minuscule : bec de 20 mm sur 9 mm, nageoire de 90 mm, 900 g.

**Sans rien exécuter**, en lisant seulement les règles de l'arbre (`rules`, affiché en 1.18), prévois sa réponse : `"Adelie"`, `"Chinstrap"`, `"Gentoo"` ou `"inconnu"` (s'il peut refuser de répondre).

a) pour l'empereur ; b) pour le poussin.

Écris ton hypothèse (cellule 📝), puis tes deux prédictions, et seulement ensuite l'**Expérience**.

In [ ]:
visitors = pd.DataFrame([[80.0, 22.0, 340.0, 30000.0],     # an emperor penguin (2 made-up measures)
                         [20.0, 9.0, 90.0, 900.0]],           # a tiny imaginary chick
                        columns=MEASURES, index=["emperor", "chick"])
visitors

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_1_19a = ...   # a) the emperor: "Adelie", "Chinstrap", "Gentoo" or "inconnu"
prediction_1_19b = ...   # b) the chick

In [ ]:
wb.check("1.19a", prediction_1_19a)
wb.check("1.19b", prediction_1_19b)

**Expérience** : exécute la cellule. `predict_proba` donne, pour chaque visiteur, la « confiance » de l'arbre pour chaque espèce (dans l'ordre de `tree.classes_`).

In [ ]:
print(tree.predict(visitors))
print(tree.classes_)
print(tree.predict_proba(visitors).round(3))

c) `proba_emperor` : la probabilité que l'arbre donne à sa réponse pour l'empereur (2 décimales).

In [ ]:
proba_emperor = ...   # c)

In [ ]:
wb.check("1.19c", proba_emperor)

### Ex 1.20 — Le score trop beau pour être vrai 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer et corriger deux erreurs qui faussent une évaluation : une fuite du label et un test vu à l'entraînement.  
**Prérequis :** Ex 1.18 · fiche §1.2.3, ⚠️ pièges · livre §1.2.3 · **Fil rouge :** Penguins · **Parcours :** C

Un collègue annonce fièrement 100 % d'accuracy sur le jeu de test avec un arbre de décision. Voici son code (cellule suivante). Ce score parfait cache **deux** erreurs. Trouve-les (indice : relis la fiche, §1.2.3 et les pièges ⚠️), puis écris `train_and_evaluate_20()`, qui fait **correctement** la même chose : un `DecisionTreeClassifier(random_state=0)` entraîné **seulement** sur le jeu d'entraînement, avec des features honnêtes (`features_20`), et évalué sur le jeu de test. Elle renvoie `(model, test_accuracy)`.

Réponds ensuite dans la cellule **📝 Mes réponses** : quelles sont les deux erreurs ? Suffit-il d'en corriger une seule ? Quelle accuracy obtiens-tu après correction (une proportion entre 0 et 1), et pourquoi est-ce une meilleure nouvelle que 100 % ?

In [ ]:
# The colleague's code (buggy!)
penguins_coded = penguins.copy()
penguins_coded["species_code"] = penguins_coded["species"].map({"Adelie": 0, "Chinstrap": 1, "Gentoo": 2})  # handy for colours
FEATURES_20 = MEASURES + ["species_code"]
model_20 = DecisionTreeClassifier(random_state=0)
model_20.fit(penguins_coded[FEATURES_20], penguins_coded["species"])
test_coded = penguins_coded.iloc[order[:100]]
print("accuracy on the test set:", model_20.score(test_coded[FEATURES_20], test_coded["species"]))

In [ ]:
features_20 = ...   # the features you keep (a list of column names)


def train_and_evaluate_20():
    """Train a DecisionTreeClassifier(random_state=0) on the training set only; return (model, test_accuracy)."""
    raise NotImplementedError("train_and_evaluate_20() is not written yet")

In [ ]:
with wb.attempt("1.20"):
    model_fixed, accuracy_fixed = train_and_evaluate_20()
    n_fit = int(model_fixed.tree_.n_node_samples[0])       # number of penguins the model was trained on
    used_20 = [str(c) for c in getattr(model_fixed, "feature_names_in_", features_20)]  # the columns it was trained on
    print(f"trained on {n_fit} penguins with the features {used_20}, test accuracy: {accuracy_fixed:.3f}")
    verdict("1.20", n_fit == len(train),
            "le modèle n'a vu que les 233 manchots d'entraînement.",
            f"le modèle a été entraîné sur {n_fit} manchots, pas sur les {len(train)} du jeu d'entraînement.")
    verdict("1.20", "species_code" not in used_20 and "species" not in used_20
            and model_fixed.n_features_in_ == len(features_20),
            "aucune feature ne contient la réponse.",
            "une de tes features contient encore la réponse (le label, sous une autre forme), ou le modèle "
            "n'a pas été entraîné sur features_20.")
    verdict("1.20", 0 < accuracy_fixed < 1,
            "un score réaliste, mesuré sur des manchots jamais vus.",
            "l'accuracy attendue est une proportion entre 0 et 1, et un score de 1 (100 %) cache encore une erreur.")

📝 **Mes réponses** (1.20) : …

## Partie C · Au-delà du supervisé : regrouper, récompenser, empiler, générer

Fiche §1.4 à §1.7. Quatre familles en boîte noire ou en version miniature : un clustering sans étiquettes, un agent qui apprend par la récompense, un réseau de neurones sur MNIST et un générateur de texte.

### Ex 1.21 — Regrouper les manchots sans leurs étiquettes 📦 ★★ ⏱️ 20 min
**Objectif :** faire tourner un clustering sans labels et comparer les groupes trouvés aux espèces.  
**Prérequis :** Ex 1.9 · 0A.52 (standardisation) · fiche §1.4.1 · livre §1.4, §1.4.1 (fig. 1.13) · **Fil rouge :** Penguins · **Parcours :** C

Oublions les espèces : l'algorithme **k-means** (détaillé au ch. 7) ne reçoit que les 4 mesures des 333 manchots et cherche 3 groupes de manchots proches les uns des autres (proches au sens de la distance entre vecteurs, 0B). On ne regarde les espèces **qu'après**, pour juger les groupes. Les trois lignes de scikit-learn sont dans l'outil fourni `cluster(X)`.

1. Écris `purity(groups, labels)` : pour chaque groupe, on compte les manchots de l'espèce la plus représentée dans ce groupe ; la pureté est la somme de ces nombres divisée par le nombre total de manchots (1 si chaque groupe ne contient qu'une espèce). Indice : `pd.crosstab(groups, labels)`, puis `.max(axis=1)`.
2. Écris `X_scaled` : les 4 mesures **mises à la même échelle**, `(X - X.mean()) / X.std()` avec `X = penguins[MEASURES]` (chaque colonne est centrée et divisée par son écart-type : la standardisation de 0A.52 ; tu verras au ch. 12 quand et pourquoi la faire).

La vérification affiche les tableaux groupes × espèces et la pureté, sur les mesures brutes puis sur les mesures mises à l'échelle. Réponds dans la cellule **📝 Mes réponses** : pourquoi les groupes sont-ils si mauvais sur les mesures brutes (pense aux unités ; voir aussi le rappel 1.R1) ? Quelles espèces restent mélangées ? Une biologiste aurait-elle pu découvrir les trois espèces de cette façon ?

In [ ]:
from sklearn.cluster import KMeans


def cluster(X):
    """Groups found by k-means (3 groups) from the measures X only: no species given."""
    kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
    return kmeans.fit_predict(X)

In [ ]:
def purity(groups, labels):
    """Sum over the groups of the size of their most frequent label, divided by the number of samples."""
    raise NotImplementedError("purity() is not written yet")


X_scaled = ...   # the 4 measures of penguins, each column centred and divided by its standard deviation

In [ ]:
with wb.attempt("1.21"):
    groups_raw = cluster(penguins[MEASURES])
    print(pd.crosstab(groups_raw, penguins["species"], rownames=["group"], colnames=["species"]))
    print(f"purity with the raw measures: {purity(groups_raw, penguins['species']):.3f}\n")
    if X_scaled is ...:
        raise NotImplementedError("X_scaled is not written yet")
    groups_scaled = cluster(X_scaled)
    print(pd.crosstab(groups_scaled, penguins["species"], rownames=["group"], colnames=["species"]))
    print(f"purity with the scaled measures: {purity(groups_scaled, penguins['species']):.3f}")
    verdict("1.21", abs(purity(np.zeros(4), np.array(["a", "a", "b", "c"])) - 0.5) < 1e-9,
            "ta pureté donne 0,5 sur un petit exemple calculable à la main (1 groupe : 2 « a » sur 4).",
            "sur un petit exemple (un seul groupe contenant a, a, b, c), la pureté devrait valoir 2/4 = 0,5.")

📝 **Mes réponses** (1.21) : …

### Ex 1.22 — L'agent cuisinier : apprendre par la récompense 🔬 ★★ ⏱️ 30 min
**Objectif :** simuler un agent qui apprend par essais et récompenses, et mesurer l'effet de la part d'exploration.  
**Prérequis :** 0A (boucles, `np.random.default_rng`) · fiche §1.6 · livre §1.6 (fig. 1.20) · **Fil rouge :** bandit · **Parcours :** C

Comme dans l'histoire du livre (§1.6), un cuisinier doit nourrir un enfant pendant un an, sans savoir ce qu'il aime. Chaque soir, il choisit une recette parmi cinq ; la seule information qu'il reçoit est si l'enfant mange (récompense 1) ou non (récompense 0). Les goûts de l'enfant sont des probabilités **cachées** (`LIKES`) : le cuisinier ne les connaît pas, il doit les découvrir en cuisinant.

Stratégie du cuisinier :
- les soirs 0 à 4 : il essaie chaque recette une fois, dans l'ordre ;
- ensuite, chaque soir, il tire `rng.random()` : si le résultat est plus petit que `explore`, il **explore** (une recette au hasard, `int(rng.integers(5))`) ; sinon il **exploite** : la recette qui a le meilleur taux de repas mangés jusqu'ici (`np.argmax`) ;
- la réaction de l'enfant est donnée par l'outil fourni `child_eats(recipe, rng)`.

Écris `cook_year(explore, rng)`, qui simule 365 soirs et renvoie `(nombre de repas mangés, liste des recettes cuisinées)`, en tirant les nombres aléatoires **exactement dans cet ordre**. La vérification contrôle d'abord une année précise (`explore = 0.1`, `rng = np.random.default_rng(0)`) : le nombre de repas mangés doit tomber pile. L'expérience fournie fait ensuite vivre 200 années à chaque valeur de `explore` ; elle trace la part des repas mangés et la part des années où, sur les 30 derniers soirs, le cuisinier cuisine le plus souvent la recette préférée de l'enfant.

Réponds dans la cellule **📝 Mes réponses** : qui est l'agent, l'environnement, l'action, la récompense ? Pourquoi `explore = 0` ne trouve-t-il pas toujours la recette préférée ? Pourquoi `explore = 1` est-il mauvais ? Quelle valeur choisirais-tu ?

In [ ]:
RECIPES = ["butter pasta", "cheese pasta", "pesto pasta", "tomato pasta", "baked potato"]
LIKES = np.array([0.6, 0.8, 0.1, 0.2, 0.5])   # hidden tastes of the child: the cook does NOT know them


def child_eats(recipe, rng):
    """The environment: 1 if the child eats the meal, else 0."""
    return int(rng.random() < LIKES[recipe])


def year_statistics(explore, n_years=200, seed=0):
    """Mean share of meals eaten, and share of years ending on the favourite recipe (last 30 evenings)."""
    rng = np.random.default_rng(seed)
    eaten, found = [], []
    for _ in range(n_years):
        n_eaten, cooked = cook_year(explore, rng)
        eaten.append(n_eaten / 365)
        found.append(np.bincount(cooked[-30:], minlength=5).argmax() == LIKES.argmax())
    return float(np.mean(eaten)), float(np.mean(found))

In [ ]:
def cook_year(explore, rng, n_evenings=365):
    """Simulate one year of dinners; returns (number of meals eaten, list of the recipes cooked)."""
    raise NotImplementedError("cook_year() is not written yet")

In [ ]:
with wb.attempt("1.22"):
    wb.check("1.22", cook_year(0.1, np.random.default_rng(0))[0])
    EXPLORES = [0, 0.05, 0.1, 0.2, 0.5, 1.0]
    stats_22 = {explore: year_statistics(explore) for explore in EXPLORES}
    for explore, (eaten, found) in stats_22.items():
        print(f"explore = {explore:<5} meals eaten: {eaten:.1%}   favourite found: {found:.0%}")
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.plot(EXPLORES, [s[0] for s in stats_22.values()], marker="o", label="share of meals eaten")
    ax.plot(EXPLORES, [s[1] for s in stats_22.values()], marker="s", label="share of years ending on the favourite")
    ax.set_xlabel("explore")
    ax.legend()
    plt.show()
    verdict("1.22", abs(stats_22[1.0][0] - LIKES.mean()) < 0.02 and stats_22[0.1][1] > 0.8,
            "en explorant tout le temps, on mange au hasard (la moyenne des goûts) ; avec 10 % d'exploration, on trouve presque toujours la recette préférée.",
            "avec explore = 1, on devrait trouver la moyenne des goûts (44 %), et avec 0.1 la recette préférée presque "
            "toujours : vérifie ton tirage au hasard et ton choix de la meilleure recette (le TAUX de repas mangés).")

📝 **Mes réponses** (1.22) : …

### Ex 1.23 — Un réseau de neurones en boîte noire sur MNIST 📦 ★★ ⏱️ 25 min
**Objectif :** entraîner un petit réseau de neurones avec scikit-learn, compter ses paramètres et examiner ses erreurs.  
**Prérequis :** Ex 1.10 · Quiz 1.Q10 · fiche §1.7 · livre §1.7 (fig. 1.21 et 1.23) · **Fil rouge :** MNIST · **Parcours :** R, C

Voici un vrai réseau de neurones (détaillé aux ch. 10 et 16) : 784 entrées (les pixels), une couche cachée de 128 neurones, 10 sorties (une par chiffre). Les trois lignes de scikit-learn sont fournies ; l'entraînement fait 30 passages (epochs) sur 5 000 images en `FAST_MODE` (60 000 sinon) et prend quelques secondes. Un avertissement `ConvergenceWarning` peut s'afficher : l'entraînement a été arrêté exprès au bout de 30 passages pour aller vite. On le laisse s'afficher : un avertissement se lit toujours (0B.37).

a) `n_params` : le nombre total de paramètres du réseau, poids et biais. Les poids sont dans la liste de matrices `mlp.coefs_`, les biais dans la liste de vecteurs `mlp.intercepts_` (additionne les `.size`). Si tu as fait l'exercice papier 1.3, compare avec ta réponse à 1.3 c.
b) `n_layers` : le nombre de couches que compte scikit-learn (`mlp.n_layers_`) ; lesquelles compte-t-il ?
c) `test_accuracy` : l'accuracy sur les 10 000 images de test (`mlp.score`, une proportion entre 0 et 1) ; et `n_errors`, le nombre d'images de test mal classées. Compare avec les 95 erreurs du petit réseau du livre (§1.7).

La vérification affiche des images de test mal classées, titrées « ✗ prédiction (vrai label) » : les aurais-tu reconnues ?

In [ ]:
from sklearn.neural_network import MLPClassifier

n_train = 5000 if FAST_MODE else 60000
X_tr, y_tr = wb.datasets.load_mnist("train", n=n_train, seed=0, flatten=True, normalize=True)   # (n_train, 784)
X_te, y_te = wb.datasets.load_mnist("test", flatten=True, normalize=True)                      # (10000, 784)

mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=30, random_state=0)   # 784 -> 128 -> 10
mlp.fit(X_tr, y_tr)                                                            # training: a few seconds
predictions_23 = mlp.predict(X_te)

In [ ]:
n_params = ...        # a) weights + biases
n_layers = ...        # b)
test_accuracy = ...   # c)
n_errors = ...        # c)

In [ ]:
wb.check("1.23a", n_params)
wb.check("1.23b", n_layers)
if test_accuracy is not ... and n_errors is not ...:
    print(f"test accuracy: {test_accuracy:.2%} · errors: {n_errors} out of {len(y_te)}")
    verdict("1.23", 0.9 <= test_accuracy <= 1 and n_errors == int((predictions_23 != y_te).sum()),
            "plus de 90 % de chiffres reconnus, sans avoir écrit une seule règle.",
            "vérifie test_accuracy (mlp.score sur X_te, y_te : une proportion entre 0 et 1) et n_errors (les prédictions différentes des labels).")
wrong = np.flatnonzero(predictions_23 != y_te)[:16]
wb.plot.show_images(X_te[wrong].reshape(-1, 28, 28), y_te[wrong], predictions_23[wrong], ncols=8)
plt.show()

### Ex 1.24 — Fabriquer du faux Holmes et du faux Verne 🔨 ★★★ ⏱️ 35 min
**Objectif :** écrire un petit générateur de texte (bigrammes de caractères) et le relier aux modèles de langage actuels.  
**Prérequis :** Ex 1.11 · 0A (`dict`, `Counter`) · fiche §1.5, 🕰️ panorama 2026 · livre §1.5 · **Fil rouge :** Holmes/Verne · **Parcours :** C

Un **générateur** (§1.5) produit de nouvelles données qui ressemblent aux exemples. Le plus simple des générateurs de texte : pour chaque caractère, on compte quels caractères le suivent dans le livre (les **bigrammes**, paires de caractères consécutifs), puis on écrit un faux texte caractère par caractère, en tirant chaque caractère suivant au hasard, avec des probabilités proportionnelles à ces comptes.

1. `bigram_counts(text)` : un dictionnaire `{caractère: Counter des caractères qui le suivent}`. Par exemple, dans `"abab"`, `"a"` est suivi 2 fois de `"b"`, et `"b"` 1 fois de `"a"`.
2. `generate(counts, start, length, rng)` : un texte de `length` caractères qui commence par le caractère `start`. À chaque pas, les candidats sont les caractères qui suivent le caractère courant, **triés** (`sorted(counts[current])`), avec les poids `counts[current][c]` ; on tire le suivant avec `rng.choice(followers, p=weights / weights.sum())`.

La vérification contrôle tes comptes (par exemple, le nombre de « qu » de *Holmes*), puis écrit 300 caractères de faux Holmes et de faux Verne. Réponds dans la cellule **📝 Mes réponses** : qu'est-ce qui ressemble à de l'anglais et à du français ? Qu'est-ce qui manque ? En quoi un grand modèle de langage (LLM) fait-il « la même chose en beaucoup plus grand » ?

In [ ]:
def bigram_counts(text):
    """For each character, a Counter of the characters that follow it in text."""
    raise NotImplementedError("bigram_counts() is not written yet")


def generate(counts, start, length, rng):
    """A text of `length` characters starting with `start`, each next character drawn from the followers
    of the current one (sorted), with probabilities proportional to their counts."""
    raise NotImplementedError("generate() is not written yet")

In [ ]:
with wb.attempt("1.24"):
    counts_holmes, counts_verne = bigram_counts(holmes), bigram_counts(verne)
    verdict("1.24", counts_holmes["q"]["u"] == holmes.count("qu")
            and sum(sum(c.values()) for c in counts_holmes.values()) == len(holmes) - 1,
            "tes comptes de bigrammes sont justes (autant de paires que de caractères moins un).",
            "tes comptes ne correspondent pas au texte : chaque paire de caractères consécutifs compte une fois.")
    fake_holmes = generate(counts_holmes, "T", 300, np.random.default_rng(0))
    fake_verne = generate(counts_verne, "L", 300, np.random.default_rng(0))
    verdict("1.24", len(fake_holmes) == 300 and fake_holmes[0] == "T",
            "le texte généré a la bonne longueur et commence par le bon caractère.",
            "le texte généré doit faire exactement `length` caractères et commencer par `start`.")
    print(fake_holmes)
    print("-" * 60)
    print(fake_verne)

📝 **Mes réponses** (1.24) : …

## Partie D · Défi final

Tout le chapitre : règles écrites à la main, jeu de test honnête, comparaison avec un modèle appris.

### Ex 1.25 — Battre l'expert : 95 % avec tes propres règles 🏆 ★★★ ⏱️ 40 min
**Objectif :** écrire ses propres règles de classification et atteindre 95 % d'accuracy sur le jeu de test, sans tricher.  
**Prérequis :** Ex 1.15, Ex 1.18 · fiche §1.1.2, §1.2.3 · **Fil rouge :** Penguins · **Parcours :** C

Les règles de la biologiste (1.15) restent sous les 95 % sur le jeu de test. À ton tour d'être l'expert : écris `my_rule(bill_length, bill_depth, flipper_length, body_mass)`, avec **au plus 4 tests** `if`/`elif` (un test peut combiner deux comparaisons avec `and`), qui atteint **au moins 95 %** d'accuracy sur le jeu de test.

**Règle du jeu (honnêteté)** : tu construis tes règles en regardant **uniquement le jeu d'entraînement** (les graphiques ci-dessous, et `accuracy(y_train, predict_with(my_rule, X_train))` autant de fois que tu veux). Tu n'évalues sur le jeu de test **qu'à la fin**, une fois tes règles fixées. Si tu ajustes tes seuils pour gagner des points sur le test, ton score ne mesure plus la généralisation (c'est l'erreur de 1.20, si tu l'as faite).

Pistes : quelle mesure sépare le mieux les Gentoo (1.15, 1.18) ? Puis, parmi les autres, laquelle sépare les Chinstrap des Adélie ?

In [ ]:
pairs = [("flipper_length_mm", "bill_length_mm"), ("bill_depth_mm", "bill_length_mm"), ("flipper_length_mm", "bill_depth_mm")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (x_name, y_name) in zip(axes, pairs):
    for species in ["Adelie", "Chinstrap", "Gentoo"]:
        part = train[train["species"] == species]
        ax.scatter(part[x_name], part[y_name], s=12, label=species)
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
axes[0].legend()
fig.suptitle("Training set only")
plt.show()

In [ ]:
def my_rule(bill_length, bill_depth, flipper_length, body_mass):
    """Your own rules (at most 4 conditions): returns "Adelie", "Chinstrap" or "Gentoo"."""
    raise NotImplementedError("my_rule() is not written yet")

In [ ]:
with wb.attempt("1.25"):
    my_train_acc = accuracy(y_train, predict_with(my_rule, X_train))
    my_test_acc = accuracy(y_test, predict_with(my_rule, X_test))
    print(f"your rules: {my_train_acc:.1%} on the training set, {my_test_acc:.1%} on the test set")
    print(pd.crosstab(y_test, predict_with(my_rule, X_test), rownames=["true"], colnames=["predicted"]))
    verdict("1.25", my_test_acc >= 0.95,
            "🏆 défi réussi : au moins 95 % sur des manchots que tes règles n'avaient jamais vus.",
            "pas encore 95 % sur le test : reviens aux graphiques du jeu d'entraînement (pas au test !).")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu décrire un dataset (échantillons, features, label) et dire à quelle famille appartient une tâche (classification, régression, clustering, renforcement, génération) ?
2. Sais-tu écrire une boucle d'entraînement minimale et expliquer ce que fait le learning rate ?
3. Sais-tu repérer une évaluation faussée (mémorisation, fuite du label, test vu à l'entraînement) ?

**Pour aller plus loin** : le *Machine Learning Crash Course* de Google et les vidéos de 3Blue1Brown sur les réseaux de neurones, cités dans la fiche. La suite : le ch. 2 (hasard et statistiques) donne les outils pour décrire les données et leurs fluctuations ; tu retrouveras le jeu de test au ch. 8, la régression au ch. 9, les neurones au ch. 10, le bandit au ch. 11, les arbres au ch. 13.